In [1]:
import pandas as pd
import numpy as np
import os

data_path = r"C:\Users\main\Desktop\Projects\Retail Demand Forecasting"

df = pd.read_csv(
    data_path + r"\outputs\retail_demand_forecasting_cleaned.csv"
)

df["Date"] = pd.to_datetime(df["Date"])

df = df.sort_values("Date").reset_index(drop=True)

print("Dataset loaded successfully!")
print("Dataset Shape:", df.shape)
df.head()

Dataset loaded successfully!
Dataset Shape: (198, 16)


,Date,Store_ID,Product_ID,Product_Category,Units_Sold,Unit_Price_NPR,Discount_Pct,Promotion,Day_of_Week,Is_Holiday,Inventory_Level,Weather,Day_of_Month,Month,Year,Weekend
0,2025-01-01,ST-01,SKU-1001,Beverages,46.0,116.88,10.0,Yes,Wednesday,0.0,72.0,Cool,1,1,2025,0
1,2025-01-02,ST-01,SKU-1001,Beverages,51.0,121.07,0.0,No,Thursday,0.0,52.0,Mild,2,1,2025,0
2,2025-01-03,ST-01,SKU-1001,Beverages,46.0,119.51,10.0,Yes,Friday,0.0,55.0,Mild,3,1,2025,0
3,2025-01-04,ST-01,SKU-1001,Beverages,42.0,121.15,0.0,No,Saturday,0.0,44.0,Mild,4,1,2025,1
4,2025-01-05,ST-01,SKU-1001,Beverages,45.0,124.30,0.0,No,Sunday,1.0,67.0,Mild,5,1,2025,1


### Creating a calender based feature

In [3]:

df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Day"] = df["Date"].dt.day


df["Day_of_Week_Number"] = df["Date"].dt.dayofweek

df["Is_Weekend"] = (
    df["Day_of_Week_Number"].isin([5, 6]).astype(int)
)


df["Week_of_Year"] = df["Date"].dt.isocalendar().week.astype(int)

print("Calendar features created!")

Calendar features created!


### Creating lag features

In [4]:
# Previous day's sales
df["Sales_Lag_1"] = df["Units_Sold"].shift(1)

# Sales from 7 days ago
df["Sales_Lag_7"] = df["Units_Sold"].shift(7)

# Sales from 14 days ago
df["Sales_Lag_14"] = df["Units_Sold"].shift(14)

print("Lag features created!")

Lag features created!


### Creating rolling average features

In [5]:
# Previous 7-day average sales
df["Rolling_Mean_7"] = (
    df["Units_Sold"]
    .shift(1)
    .rolling(window=7, min_periods=3)
    .mean()
)

# Previous 14-day average sales
df["Rolling_Mean_14"] = (
    df["Units_Sold"]
    .shift(1)
    .rolling(window=14, min_periods=7)
    .mean()
)

print("Rolling average features created!")

Rolling average features created!


In [7]:
new_features = [
    "Date",
    "Units_Sold",
    "Year",
    "Month",
    "Day",
    "Day_of_Week_Number",
    "Is_Weekend",
    "Week_of_Year",
    "Sales_Lag_1",
    "Sales_Lag_7",
    "Sales_Lag_14",
    "Rolling_Mean_7",
    "Rolling_Mean_14"
]

df[new_features].head(6)

,Date,Units_Sold,Year,Month,Day,Day_of_Week_Number,Is_Weekend,Week_of_Year,Sales_Lag_1,Sales_Lag_7,Sales_Lag_14,Rolling_Mean_7,Rolling_Mean_14
0,2025-01-01,46.0,2025,1,1,2,0,1,NaN,NaN,NaN,NaN,NaN
1,2025-01-02,51.0,2025,1,2,3,0,1,46.0,NaN,NaN,NaN,NaN
2,2025-01-03,46.0,2025,1,3,4,0,1,51.0,NaN,NaN,NaN,NaN
3,2025-01-04,42.0,2025,1,4,5,1,1,46.0,NaN,NaN,47.666667,NaN
4,2025-01-05,45.0,2025,1,5,6,1,1,42.0,NaN,NaN,46.250000,NaN
5,2025-01-06,47.0,2025,1,6,0,0,2,45.0,NaN,NaN,46.000000,NaN


In [8]:
feature_columns = [
    "Sales_Lag_1",
    "Sales_Lag_7",
    "Sales_Lag_14",
    "Rolling_Mean_7",
    "Rolling_Mean_14"
]

print("Missing values in engineered features:")
print(df[feature_columns].isnull().sum())

Missing values in engineered features:
Sales_Lag_1         7
Sales_Lag_7        13
Sales_Lag_14       20
Rolling_Mean_7      3
Rolling_Mean_14     7
dtype: int64


In [10]:

output_path = data_path + r"\outputs"
os.makedirs(output_path, exist_ok=True)

df.to_csv(
    output_path + r"\retail_demand_forecasting_featured.csv",
    index=False
)

print("Feature-engineered dataset saved successfully!")
print("Final Dataset Shape:", df.shape)

Feature-engineered dataset saved successfully!
Final Dataset Shape: (198, 25)
